In [1]:
import pandas as pd

In [3]:
muestra_validada = pd.read_parquet(
    r"C:\Users\Usuario\TFM\TFM_scraping_contratacion_estado\data\silver\muestra_10_estratificada_3_portales.parquet"
)

muestra_validada["portal"].value_counts()

portal
contratacion_estado    4
galicia                3
madrid                 3
Name: count, dtype: int64

In [4]:
# ============================================================
# 2. Separar contratación_estado
# ============================================================

muestra_contratacion = muestra_validada[
    muestra_validada["portal"] == "contratacion_estado"
].copy()

muestra_contratacion[[
    "licitacion_id",
    "portal",
    "titulo",
    "detail_url"
]]

,licitacion_id,portal,titulo,detail_url
6,41db3a58bd5c3bbd,contratacion_estado,Servicio Especializado de Rehabilitación Hospi...,https://contrataciondelestado.es/wps/poc?uri=d...
7,1967eade2fccb129,contratacion_estado,Contratación del servicio de Asistencia Sanita...,https://contrataciondelestado.es/wps/portal/%2...
8,bcf984626102e175,contratacion_estado,Servicio de prevención ajeno en las especialid...,https://contrataciondelestado.es/wps/poc?uri=d...
9,ddec9f7e2e9944dc,contratacion_estado,Servicio de reconocimientos médicos personal a...,https://contrataciondelestado.es/FileSystem/se...


In [11]:
# ============================================================
# 3. Clasificar URLs de contratación_estado
# ============================================================

muestra_contratacion["es_pdf_directo"] = (
    muestra_contratacion["detail_url"]
    .str.contains("GetDocumentByIdServlet", case=False, na=False)
)

muestra_contratacion[[
    "licitacion_id",
    "titulo",
    "detail_url",
    "es_pdf_directo"
]]

,licitacion_id,titulo,detail_url,es_pdf_directo
6,41db3a58bd5c3bbd,Servicio Especializado de Rehabilitación Hospi...,https://contrataciondelestado.es/wps/poc?uri=d...,False
7,1967eade2fccb129,Contratación del servicio de Asistencia Sanita...,https://contrataciondelestado.es/wps/portal/%2...,False
8,bcf984626102e175,Servicio de prevención ajeno en las especialid...,https://contrataciondelestado.es/wps/poc?uri=d...,False
9,ddec9f7e2e9944dc,Servicio de reconocimientos médicos personal a...,https://contrataciondelestado.es/FileSystem/se...,True


In [9]:
muestra_contratacion.iloc[3]["detail_url"]

'https://contrataciondelestado.es/FileSystem/servlet/GetDocumentByIdServlet?DocumentIdParam=DYMMp2mFqXz4PkjvZh9I5vyWLItwtIt0n8yxEgEwqVaUFQkv5PXRMxoYGegc0NMYm0UHT%2FoXEFKTkr37fnFcEQrX2pafwfyc3CjQTJA0783VTD3T98KC0nSgQFM0q%2B5s&cifrado=QUC1GjXXSiLkydRHJBmbpw%3D%3D'

In [12]:
# ============================================================
# 4. Quedarnos solo con HTML de contratación_estado
# ============================================================

muestra_contratacion_html = muestra_contratacion[
    ~muestra_contratacion["es_pdf_directo"]
].copy()

muestra_contratacion_html[[
    "licitacion_id",
    "titulo",
    "detail_url"
]]

,licitacion_id,titulo,detail_url
6,41db3a58bd5c3bbd,Servicio Especializado de Rehabilitación Hospi...,https://contrataciondelestado.es/wps/poc?uri=d...
7,1967eade2fccb129,Contratación del servicio de Asistencia Sanita...,https://contrataciondelestado.es/wps/portal/%2...
8,bcf984626102e175,Servicio de prevención ajeno en las especialid...,https://contrataciondelestado.es/wps/poc?uri=d...


In [13]:
# ============================================================
# 5. Seleccionar una licitación HTML individual
# ============================================================

licitacion_html = muestra_contratacion_html.iloc[0].copy()

licitacion_id_html = licitacion_html["licitacion_id"]
url_html = licitacion_html["detail_url"]

print("Licitación:", licitacion_id_html)
print("Título:", licitacion_html["titulo"])
print("URL:", url_html)

Licitación: 41db3a58bd5c3bbd
Título: Servicio Especializado de Rehabilitación Hospitalaria en el ámbito territorial de Mérida (Badajoz) y área de influencia
URL: https://contrataciondelestado.es/wps/poc?uri=deeplink:detalle_licitacion&idEvl=rsl%2BfImh9a6LAncw3qdZkA%3D%3D


In [14]:
# ============================================================
# 5. Seleccionar una licitación HTML individual
# ============================================================

licitacion_html = muestra_contratacion_html.iloc[0].copy()

licitacion_id_html = licitacion_html["licitacion_id"]
url_html = licitacion_html["detail_url"]

print("Licitación:", licitacion_id_html)
print("Título:", licitacion_html["titulo"])
print("URL:", url_html)

Licitación: 41db3a58bd5c3bbd
Título: Servicio Especializado de Rehabilitación Hospitalaria en el ámbito territorial de Mérida (Badajoz) y área de influencia
URL: https://contrataciondelestado.es/wps/poc?uri=deeplink:detalle_licitacion&idEvl=rsl%2BfImh9a6LAncw3qdZkA%3D%3D


In [15]:
# ============================================================
# 6. Descargar HTML
# ============================================================

import requests
import pandas as pd
import re

from bs4 import BeautifulSoup
from urllib.parse import urljoin

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

response = requests.get(
    url_html,
    headers=headers,
    timeout=30
)

print("Status:", response.status_code)
print("URL final:", response.url)
print("Content-Type:", response.headers.get("Content-Type"))
print("Tamaño HTML:", len(response.text))

Status: 200
URL final: https://contrataciondelestado.es/wps/portal/plataforma/buscadores/detalle/!ut/p/z1/hU_JDoIwFPwi09cCrRxZSimCgCxKL4TExJCwGGP4fovhqrzbZJY3gxS6ITV1S__o3v08dYPGjaKtyVPPC0ICx8LwgcR-VdFwhQTV6LonUZqGH-eA9quvxDI8s47qjBZSAMgw8OMKWyAI3QR_MhrdgbVOzXNH2gak7kV3iLKkzATBABRV0_wa9Z5izervfBlQQzRBmcVsc_MLARK7Akx8OvtAM8Yjnpfriz3_cwxseWjGNimdD2Jmwiw!/dz/d5/L2dBISEvZ0FBIS9nQSEh/
Content-Type: text/html; charset=UTF-8
Tamaño HTML: 130714


In [16]:
# ============================================================
# 7. Validar tipo de contenido
# ============================================================

content_type = response.headers.get("Content-Type", "").lower()

print("Content-Type:", content_type)
print("Primeros bytes:", response.content[:20])

if response.content.startswith(b"%PDF"):
    print("Es PDF directo")
elif "html" in content_type or "<html" in response.text.lower():
    print("Es HTML")
else:
    print("Tipo no identificado")

Content-Type: text/html; charset=utf-8
Primeros bytes: b'<!DOCTYPE html>\n<htm'
Es HTML


In [17]:
# ============================================================
# 8. Parsear HTML
# ============================================================

soup = BeautifulSoup(response.text, "html.parser")

titulo_pagina = soup.title.get_text(strip=True) if soup.title else None

print("Título página:")
print(titulo_pagina)

Título página:
Plataforma de Contratación del Sector Público


In [24]:


# ============================================================
# 10. Extraer texto visible por líneas
# ============================================================

texto_html = soup.get_text(
    separator="\n",
    strip=True
)

lineas_html = [
    linea.strip()
    for linea in texto_html.splitlines()
    if linea.strip()
]

print("Número de líneas:", len(lineas_html))

for i, linea in enumerate(lineas_html[:300]):
    print(f"{i}: {linea}")

Número de líneas: 174
0: Plataforma de Contratación del Sector Público
1: Detail
2: EN |
3: Bienvenidos
4: Ongi Etorri
5: Benvinguts
6: Benvidos
7: Welcome
8: Bienvenue
9: Search
10: Contractor Profile
11: Companies
12: Public Organizations
13: Verify CSV
14: Statistics
15: Open Data
16: Contact
17: Menú
18: Cerrar
19: Inicio
20: Search
21: Contractor Profile
22: Companies
23: Public Organizations
24: Verify CSV
25: Statistics
26: Open Data
27: Contact
28: Search
29: Detail
30: SeguimientoExpedientePortlet
31: If you would like to be notified of any news regarding this bid, please register on the State Contracting Platform homepage.
32: Start session
33: Register
34: Confirmación de Eliminación
35: The bid with the file number
36: ¿Desea continuar?
37: Su navegador no permite scripts o están desactivados
38: Details of tender
39: Su navegador no permite scripts o están desactivados
40: Contracting Party
41: Dirección General de la Mutual Midat Cyclops, Mutua Colaboradora con la Segurid

In [25]:
# ============================================================
# 11. Buscar etiquetas exactas del formato HTML contratación_estado
# ============================================================

etiquetas_contratacion_html = [
    "Órgano de contratación",
    "Organo de contratación",
    "Expediente",
    "Objeto del contrato",
    "Enlace a la licitación",
    "Estado de la Licitación",
    "Valor estimado del contrato",
    "Tipo de Contrato",
    "Código CPV",
    "Lugar de ejecución",
    "Sistema de contratación",
    "Procedimiento de contratación",
    "Tipo de tramitación",
    "Método de presentación de la oferta",
    "Fecha fin de presentación de oferta",
    "Anuncios y Documentos",
    "Publicación en plataforma",
    "Documento",
    "Ver documentos"
]

for etiqueta in etiquetas_contratacion_html:
    print("\n" + "=" * 80)
    print("Etiqueta:", etiqueta)

    encontrados = [
        (i, linea)
        for i, linea in enumerate(lineas_html)
        if etiqueta.lower() in linea.lower()
    ]

    for i, linea in encontrados[:10]:
        print(f"{i}: {linea}")


Etiqueta: Órgano de contratación
104: Diligencia Órgano de Contratación

Etiqueta: Organo de contratación

Etiqueta: Expediente
30: SeguimientoExpedientePortlet
94: Documento de aprobación del expediente

Etiqueta: Objeto del contrato

Etiqueta: Enlace a la licitación

Etiqueta: Estado de la Licitación

Etiqueta: Valor estimado del contrato

Etiqueta: Tipo de Contrato

Etiqueta: Código CPV

Etiqueta: Lugar de ejecución

Etiqueta: Sistema de contratación

Etiqueta: Procedimiento de contratación

Etiqueta: Tipo de tramitación

Etiqueta: Método de presentación de la oferta

Etiqueta: Fecha fin de presentación de oferta

Etiqueta: Anuncios y Documentos

Etiqueta: Publicación en plataforma

Etiqueta: Documento
87: Otros Documentos
94: Documento de aprobación del expediente

Etiqueta: Ver documentos


In [26]:
# ============================================================
# 12. Buscar líneas clave generales
# ============================================================

palabras_clave = [
    "Expediente",
    "Órgano",
    "Organo",
    "Contratación",
    "Objeto",
    "CPV",
    "Presupuesto",
    "Valor estimado",
    "Procedimiento",
    "Tramitación",
    "Presentación",
    "Fecha",
    "Importe",
    "Adjudicatario",
    "Estado",
    "Lugar",
    "NUTS",
    "Contrato",
    "Licitación",
    "Plazo",
    "Documentos",
    "Anuncio",
    "Clasificación",
    "Tipo de contrato",
    "Resultado",
    "Formalización"
]

lineas_clave = []

for i, linea in enumerate(lineas_html):
    if any(palabra.lower() in linea.lower() for palabra in palabras_clave):
        lineas_clave.append({
            "indice": i,
            "linea": linea
        })

df_lineas_clave = pd.DataFrame(lineas_clave)

df_lineas_clave.head(150)

,indice,linea
0,0,Plataforma de Contratación del Sector Público
1,30,SeguimientoExpedientePortlet
2,49,https://contrataciondelestado.es/wps/poc?uri=d...
3,62,CPV code
4,82,Rectificación de Anuncio de Licitación
5,84,Anuncio de Licitación
6,87,Otros Documentos
7,94,Documento de aprobación del expediente
8,96,Acta del órgano de asistencia
9,104,Diligencia Órgano de Contratación


In [27]:
# ============================================================
# 13. Imprimir contexto alrededor de líneas clave
# ============================================================

indices_clave = df_lineas_clave["indice"].tolist()

indices_filtrados = []

for idx in indices_clave:
    if not indices_filtrados or idx - indices_filtrados[-1] > 8:
        indices_filtrados.append(idx)

for idx in indices_filtrados[:40]:
    print("\n" + "=" * 100)
    print(f"Contexto alrededor de línea {idx}")
    print("=" * 100)

    inicio = max(idx - 6, 0)
    fin = min(idx + 18, len(lineas_html))

    for j in range(inicio, fin):
        print(f"{j}: {lineas_html[j]}")


Contexto alrededor de línea 0
0: Plataforma de Contratación del Sector Público
1: Detail
2: EN |
3: Bienvenidos
4: Ongi Etorri
5: Benvinguts
6: Benvidos
7: Welcome
8: Bienvenue
9: Search
10: Contractor Profile
11: Companies
12: Public Organizations
13: Verify CSV
14: Statistics
15: Open Data
16: Contact
17: Menú

Contexto alrededor de línea 30
24: Verify CSV
25: Statistics
26: Open Data
27: Contact
28: Search
29: Detail
30: SeguimientoExpedientePortlet
31: If you would like to be notified of any news regarding this bid, please register on the State Contracting Platform homepage.
32: Start session
33: Register
34: Confirmación de Eliminación
35: The bid with the file number
36: ¿Desea continuar?
37: Su navegador no permite scripts o están desactivados
38: Details of tender
39: Su navegador no permite scripts o están desactivados
40: Contracting Party
41: Dirección General de la Mutual Midat Cyclops, Mutua Colaboradora con la Seguridad Social, nº 1
42: 50254410043881
43: OTRAS ENTIDADES

In [28]:
# ============================================================
# 14. Revisar tablas HTML
# ============================================================

tablas = soup.find_all("table")

print("Número de tablas:", len(tablas))

for i, tabla in enumerate(tablas[:15], start=1):
    filas = tabla.find_all("tr")

    print("\n" + "=" * 100)
    print(f"Tabla {i} | filas: {len(filas)}")
    print("=" * 100)

    for fila in filas[:15]:
        celdas = [
            celda.get_text(" ", strip=True)
            for celda in fila.find_all(["th", "td"])
        ]

        if celdas:
            print(celdas)

Número de tablas: 9

Tabla 1 | filas: 4
['The bid with the file number ¿Desea continuar?', 'The bid with the file number ¿Desea continuar?', '', 'The bid with the file number', '', '', '¿Desea continuar?', '']
['The bid with the file number ¿Desea continuar?', '', 'The bid with the file number', '', '', '¿Desea continuar?', '']
['', 'The bid with the file number', '']
['', '¿Desea continuar?', '']

Tabla 2 | filas: 3
['The bid with the file number ¿Desea continuar?', '', 'The bid with the file number', '', '', '¿Desea continuar?', '']
['', 'The bid with the file number', '']
['', '¿Desea continuar?', '']

Tabla 3 | filas: 2
['', 'The bid with the file number', '']
['', '¿Desea continuar?', '']

Tabla 4 | filas: 1
['', '', '']

Tabla 5 | filas: 4
['Contracting Party Dirección General de la Mutual Midat Cyclops, Mutua Colaboradora con la Seguridad Social, nº 1 50254410043881 OTRAS ENTIDADES DEL SECTOR PÚBLICO>MUTUAS DE ACCIDENTES DE TRABAJO COLABORADORAS DE LA SEGURIDAD SOCIAL>Mutua Mida

In [29]:
# ============================================================
# 15. Convertir tablas HTML a DataFrames
# ============================================================

dfs_tablas = []

for i, tabla in enumerate(tablas, start=1):
    filas_tabla = []

    for fila in tabla.find_all("tr"):
        celdas = [
            celda.get_text(" ", strip=True)
            for celda in fila.find_all(["th", "td"])
        ]

        if celdas:
            filas_tabla.append(celdas)

    if filas_tabla:
        max_cols = max(len(fila) for fila in filas_tabla)

        filas_normalizadas = [
            fila + [None] * (max_cols - len(fila))
            for fila in filas_tabla
        ]

        df_tabla = pd.DataFrame(filas_normalizadas)
        df_tabla["num_tabla"] = i
        dfs_tablas.append(df_tabla)

print("Tablas convertidas:", len(dfs_tablas))

for i, df_tabla in enumerate(dfs_tablas[:10], start=1):
    print("\n" + "=" * 100)
    print(f"DF tabla {i}")
    print("=" * 100)
    display(df_tabla.head(20))

Tablas convertidas: 9

DF tabla 1


,0,1,2,3,4,5,6,7,num_tabla
0,The bid with the file number ¿Desea continuar?,The bid with the file number ¿Desea continuar?,,The bid with the file number,,,¿Desea continuar?,,1
1,The bid with the file number ¿Desea continuar?,,The bid with the file number,,,¿Desea continuar?,,None,1
2,,The bid with the file number,,None,None,None,None,None,1
3,,¿Desea continuar?,,None,None,None,None,None,1



DF tabla 2


,0,1,2,3,4,5,6,num_tabla
0,The bid with the file number ¿Desea continuar?,,The bid with the file number,,,¿Desea continuar?,,2
1,,The bid with the file number,,None,None,None,None,2
2,,¿Desea continuar?,,None,None,None,None,2



DF tabla 3


...

,0,num_tabla
0,Method of presenting the offer Electrónica,7
1,End date for the submission of offers 04/05/20...,7



DF tabla 8


,0,1,2,num_tabla
0,Post on platform,Document,Veure documents,8
1,15/04/2026 07:59:27,Rectificación de Anuncio de Licitación,,8
2,15/04/2026 07:59:01,Anuncio de Licitación,,8
3,15/04/2026 07:59:48,Pliego,,8



DF tabla 9


,0,1,2,num_tabla
0,Post on platform,Document,Veure documents,9
1,15/04/2026 08:00:12,Memoria justificativa,,9
2,15/04/2026 08:00:17,Documento de aprobación del expediente,,9
3,06/05/2026 14:38:25,Acta del órgano de asistencia,,9
4,13/05/2026 10:44:44,OFERTA TECNICA,,9
5,13/05/2026 10:44:46,Ofertas economicas,,9
6,19/05/2026 07:45:23,Informe de valoración de los criterios de adju...,,9
7,19/05/2026 07:45:26,Diligencia Órgano de Contratación,,9
8,19/05/2026 07:45:28,Acta del órgano de asistencia,,9


In [30]:
# ============================================================
# Revisar bloque útil de la ficha HTML
# ============================================================

for i, linea in enumerate(lineas_html[20:120], start=20):
    print(f"{i}: {linea}")

20: Search
21: Contractor Profile
22: Companies
23: Public Organizations
24: Verify CSV
25: Statistics
26: Open Data
27: Contact
28: Search
29: Detail
30: SeguimientoExpedientePortlet
31: If you would like to be notified of any news regarding this bid, please register on the State Contracting Platform homepage.
32: Start session
33: Register
34: Confirmación de Eliminación
35: The bid with the file number
36: ¿Desea continuar?
37: Su navegador no permite scripts o están desactivados
38: Details of tender
39: Su navegador no permite scripts o están desactivados
40: Contracting Party
41: Dirección General de la Mutual Midat Cyclops, Mutua Colaboradora con la Seguridad Social, nº 1
42: 50254410043881
43: OTRAS ENTIDADES DEL SECTOR PÚBLICO>MUTUAS DE ACCIDENTES DE TRABAJO COLABORADORAS DE LA SEGURIDAD SOCIAL>Mutua Midat Cyclops
44: File
45: N202600214
46: Subject of the contract
47: Servicio Especializado de Rehabilitación Hospitalaria en el ámbito territorial de Mérida (Badajoz) y área de 

In [31]:
# ============================================================
# 2. Parser ajustado para HTML contratación_estado
#    con etiquetas en español e inglés
# ============================================================

import re
import pandas as pd


ETIQUETAS_CORTE_CONTRATACION_HTML = [
    # Español
    "Órgano de contratación",
    "Organo de contratación",
    "Expediente",
    "Objeto del contrato",
    "Enlace a la licitación",
    "Estado de la Licitación",
    "Valor estimado del contrato",
    "Tipo de Contrato",
    "Código CPV",
    "Lugar de ejecución",
    "Sistema de contratación",
    "Procedimiento de contratación",
    "Tipo de tramitación",
    "Método de presentación de la oferta",
    "Fecha fin de presentación de oferta",
    "Anuncios y Documentos",
    "Otra información",
    "Publicación en plataforma",
    "Documento",
    "Ver documentos",

    # Inglés / etiquetas internas de PLACSP
    "Details of tender",
    "Contracting Party",
    "File",
    "Tender status",
    "Estimated value",
    "Type of Contract",
    "CPV code",
    "Place of execution",
    "Contracting system",
    "Contracting procedure",
    "Processing type",
    "Method of presentation of the offer",
    "Deadline for submission of tenders",
    "Announcements and Documents",
    "Publication on platform",
]


def limpiar_texto_html(valor):
    """
    Limpieza básica de texto.
    """
    if valor is None or pd.isna(valor):
        return None

    valor = str(valor)
    valor = re.sub(r"\s+", " ", valor).strip()

    return valor


def es_etiqueta_corte(linea):
    """
    Indica si una línea corresponde a una etiqueta conocida.
    """
    linea_norm = linea.lower().strip()

    return linea_norm in [
        etiqueta.lower().strip()
        for etiqueta in ETIQUETAS_CORTE_CONTRATACION_HTML
    ]


def buscar_indice_etiqueta(lineas, etiquetas):
    """
    Busca el índice de la primera etiqueta encontrada.
    etiquetas puede ser string o lista.
    """
    if isinstance(etiquetas, str):
        etiquetas = [etiquetas]

    etiquetas_norm = [
        etiqueta.lower().strip()
        for etiqueta in etiquetas
    ]

    for i, linea in enumerate(lineas):
        linea_norm = linea.lower().strip()

        if linea_norm in etiquetas_norm:
            return i

    return None


def extraer_bloque_despues_etiqueta(
    lineas,
    etiquetas_inicio,
    max_lineas=8
):
    """
    Extrae las líneas posteriores a una etiqueta hasta encontrar otra etiqueta.
    """

    idx = buscar_indice_etiqueta(lineas, etiquetas_inicio)

    if idx is None:
        return None

    valores = []

    for j in range(idx + 1, min(idx + 1 + max_lineas, len(lineas))):
        linea = lineas[j].strip()

        if not linea:
            continue

        if es_etiqueta_corte(linea):
            break

        # Filtrar ruido muy claro
        if linea in ["EN", "Menú", "Detail", "Search"]:
            continue

        if "Su navegador no permite scripts" in linea:
            continue

        if "your browser" in linea.lower():
            continue

        valores.append(linea)

    if not valores:
        return None

    return limpiar_texto_html(" ".join(valores))


def convertir_importe_europeo(valor):
    """
    Convierte importes europeos.
    Ejemplo:
        '80.366,00 Euros' -> 80366.0
    """
    if valor is None or pd.isna(valor):
        return None

    match = re.search(r"([0-9\.\,]+)", str(valor))

    if not match:
        return None

    numero = match.group(1)
    numero = numero.replace(".", "")
    numero = numero.replace(",", ".")

    try:
        return float(numero)
    except ValueError:
        return None


def extraer_cpv(valor):
    """
    Extrae código y descripción CPV.
    Ejemplo:
        '85100000-Servicios de salud.'
    """
    if valor is None:
        return None, None

    match = re.search(r"(\d{8})\s*[-–]\s*(.+)", str(valor))

    if match:
        return match.group(1), limpiar_texto_html(match.group(2))

    match_codigo = re.search(r"\b\d{8}\b", str(valor))

    if match_codigo:
        return match_codigo.group(0), None

    return None, None


def extraer_primer_url(lineas, contiene=None):
    """
    Extrae primera URL. Si contiene no es None, filtra por substring.
    """
    for linea in lineas:
        if "http" in linea:
            if contiene is None or contiene.lower() in linea.lower():
                return linea.strip()

    return None

In [32]:
# ============================================================
# 3. Extracción de campos principales
# ============================================================

organo_contratacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Órgano de contratación", "Organo de contratación", "Contracting Party"],
    max_lineas=4
)

expediente = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Expediente", "File"],
    max_lineas=2
)

objeto_contrato = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Objeto del contrato"],
    max_lineas=4
)

# Si no encuentra objeto por etiqueta, intentamos por el título original de la muestra
if objeto_contrato is None:
    objeto_contrato = licitacion_html.get("titulo")

enlace_licitacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Enlace a la licitación"],
    max_lineas=4
)

if enlace_licitacion is None:
    enlace_licitacion = extraer_primer_url(
        lineas_html,
        contiene="detalle_licitacion"
    )

estado_licitacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Estado de la Licitación", "Tender status"],
    max_lineas=2
)

valor_estimado_texto = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Valor estimado del contrato", "Estimated value"],
    max_lineas=2
)

tipo_contrato = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Tipo de Contrato", "Type of Contract"],
    max_lineas=2
)

codigo_cpv_texto = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Código CPV", "CPV code"],
    max_lineas=2
)

codigo_cpv, descripcion_cpv = extraer_cpv(codigo_cpv_texto)

lugar_ejecucion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Lugar de ejecución", "Place of execution"],
    max_lineas=2
)

sistema_contratacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Sistema de contratación", "Contracting system"],
    max_lineas=2
)

procedimiento_contratacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Procedimiento de contratación", "Contracting procedure"],
    max_lineas=2
)

tipo_tramitacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Tipo de tramitación", "Processing type"],
    max_lineas=2
)

metodo_presentacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Método de presentación de la oferta", "Method of presentation of the offer"],
    max_lineas=2
)

fecha_fin_presentacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Fecha fin de presentación de oferta", "Deadline for submission of tenders"],
    max_lineas=2
)

In [33]:
# ============================================================
# 4. Crear ficha HTML contratación_estado
# ============================================================

ficha_html_contratacion_estado = {
    "licitacion_id": licitacion_id_html,
    "portal": "contratacion_estado",
    "tipo_fuente": "html_contratacion_estado",
    "url_original": url_html,
    "url_final": response.url,
    "titulo_original": licitacion_html.get("titulo"),
    "titulo_pagina": titulo_pagina,

    "organo_contratacion": organo_contratacion,
    "expediente": expediente,
    "objeto_contrato": objeto_contrato,
    "enlace_licitacion": enlace_licitacion,
    "estado_licitacion": estado_licitacion,

    "valor_estimado_texto": valor_estimado_texto,
    "valor_estimado_num": convertir_importe_europeo(valor_estimado_texto),

    "tipo_contrato": tipo_contrato,
    "codigo_cpv_texto": codigo_cpv_texto,
    "codigo_cpv": codigo_cpv,
    "descripcion_cpv": descripcion_cpv,

    "lugar_ejecucion": lugar_ejecucion,
    "sistema_contratacion": sistema_contratacion,
    "procedimiento_contratacion": procedimiento_contratacion,
    "tipo_tramitacion": tipo_tramitacion,
    "metodo_presentacion": metodo_presentacion,
    "fecha_fin_presentacion": fecha_fin_presentacion,

    "num_lineas_extraidas": len(lineas_html)
}

df_ficha_html_contratacion_estado = pd.DataFrame([
    ficha_html_contratacion_estado
])

df_ficha_html_contratacion_estado.T

,0
licitacion_id,41db3a58bd5c3bbd
portal,contratacion_estado
tipo_fuente,html_contratacion_estado
url_original,https://contrataciondelestado.es/wps/poc?uri=d...
url_final,https://contrataciondelestado.es/wps/portal/pl...
titulo_original,Servicio Especializado de Rehabilitación Hospi...
titulo_pagina,Plataforma de Contratación del Sector Público
organo_contratacion,"Dirección General de la Mutual Midat Cyclops, ..."
expediente,N202600214 Subject of the contract
objeto_contrato,Servicio Especializado de Rehabilitación Hospi...


In [34]:
# ============================================================
# 1. Ampliar etiquetas de corte español / inglés
# ============================================================

ETIQUETAS_CORTE_CONTRATACION_HTML = [
    # Español
    "Órgano de contratación",
    "Organo de contratación",
    "Expediente",
    "Objeto del contrato",
    "Enlace a la licitación",
    "Estado de la Licitación",
    "Valor estimado del contrato",
    "Tipo de Contrato",
    "Código CPV",
    "Lugar de ejecución",
    "Sistema de contratación",
    "Procedimiento de contratación",
    "Tipo de tramitación",
    "Método de presentación de la oferta",
    "Fecha fin de presentación de oferta",
    "Otra información",
    "Anuncios y Documentos",
    "Publicación en plataforma",
    "Documento",
    "Ver documentos",

    # Inglés / PLACSP
    "Details of tender",
    "Contracting Party",
    "File",
    "Subject of the contract",
    "Link to tender",
    "Tender status",
    "Estimated value of the contract",
    "Estimated value",
    "Type of Contract",
    "CPV code",
    "Place of execution",
    "Contracting system",
    "Contracting procedure",
    "Processing type",
    "Method of presentation of the offer",
    "Deadline for submission of tenders",
    "Other information",
    "Announcements and Documents",
    "Publication on platform",
    "Document",
    "View documents",
]

In [35]:
# ============================================================
# 2. Funciones ajustadas de extracción
# ============================================================

def limpiar_texto_html(valor):
    if valor is None or pd.isna(valor):
        return None

    valor = str(valor)
    valor = re.sub(r"\s+", " ", valor).strip()

    return valor


def es_etiqueta_corte(linea):
    linea_norm = linea.lower().strip()

    etiquetas_norm = [
        etiqueta.lower().strip()
        for etiqueta in ETIQUETAS_CORTE_CONTRATACION_HTML
    ]

    return linea_norm in etiquetas_norm


def buscar_indice_etiqueta(lineas, etiquetas):
    if isinstance(etiquetas, str):
        etiquetas = [etiquetas]

    etiquetas_norm = [
        etiqueta.lower().strip()
        for etiqueta in etiquetas
    ]

    for i, linea in enumerate(lineas):
        linea_norm = linea.lower().strip()

        if linea_norm in etiquetas_norm:
            return i

    return None


def extraer_bloque_despues_etiqueta(
    lineas,
    etiquetas_inicio,
    max_lineas=8
):
    idx = buscar_indice_etiqueta(lineas, etiquetas_inicio)

    if idx is None:
        return None

    valores = []

    for j in range(idx + 1, min(idx + 1 + max_lineas, len(lineas))):
        linea = lineas[j].strip()

        if not linea:
            continue

        if es_etiqueta_corte(linea):
            break

        # Ruido del portal
        if linea in ["EN", "Menú", "Detail", "Search"]:
            continue

        if "Su navegador no permite scripts" in linea:
            continue

        if "your browser" in linea.lower():
            continue

        if "¿Desea continuar?" in linea:
            continue

        if "The bid with the file number" in linea:
            continue

        valores.append(linea)

    if not valores:
        return None

    return limpiar_texto_html(" ".join(valores))


def limpiar_expediente(valor):
    """
    Corrige casos como:
    'N202600214 Subject of the contract'
    """
    if valor is None:
        return None

    match = re.search(r"\b[A-Z]?\d{6,}[A-Z0-9/\-]*\b", str(valor))

    if match:
        return match.group(0)

    return limpiar_texto_html(valor)


def limpiar_lugar_ejecucion(valor):
    """
    Corrige casos donde se pega la siguiente etiqueta.
    """
    if valor is None:
        return None

    valor = str(valor)

    cortes = [
        "Sistema de contratación",
        "Contracting system",
        "Procedimiento de contratación",
        "Contracting procedure",
    ]

    for corte in cortes:
        if corte in valor:
            valor = valor.split(corte)[0]

    return limpiar_texto_html(valor)


def convertir_importe_europeo(valor):
    if valor is None or pd.isna(valor):
        return None

    match = re.search(r"([0-9\.\,]+)", str(valor))

    if not match:
        return None

    numero = match.group(1)
    numero = numero.replace(".", "")
    numero = numero.replace(",", ".")

    try:
        return float(numero)
    except ValueError:
        return None


def extraer_cpv(valor):
    if valor is None:
        return None, None

    match = re.search(r"(\d{8})\s*[-–]\s*(.+)", str(valor))

    if match:
        return match.group(1), limpiar_texto_html(match.group(2))

    match_codigo = re.search(r"\b\d{8}\b", str(valor))

    if match_codigo:
        return match_codigo.group(0), None

    return None, None

In [36]:
# ============================================================
# 3. Reextraer campos principales
# ============================================================

organo_contratacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Órgano de contratación", "Organo de contratación", "Contracting Party"],
    max_lineas=4
)

expediente = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Expediente", "File"],
    max_lineas=3
)

expediente = limpiar_expediente(expediente)

objeto_contrato = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Objeto del contrato", "Subject of the contract"],
    max_lineas=5
)

if objeto_contrato is None:
    objeto_contrato = licitacion_html.get("titulo")

enlace_licitacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Enlace a la licitación", "Link to tender"],
    max_lineas=4
)

estado_licitacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Estado de la Licitación", "Tender status"],
    max_lineas=2
)

valor_estimado_texto = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Valor estimado del contrato", "Estimated value of the contract", "Estimated value"],
    max_lineas=2
)

tipo_contrato = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Tipo de Contrato", "Type of Contract"],
    max_lineas=2
)

codigo_cpv_texto = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Código CPV", "CPV code"],
    max_lineas=2
)

codigo_cpv, descripcion_cpv = extraer_cpv(codigo_cpv_texto)

lugar_ejecucion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Lugar de ejecución", "Place of execution"],
    max_lineas=3
)

lugar_ejecucion = limpiar_lugar_ejecucion(lugar_ejecucion)

sistema_contratacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Sistema de contratación", "Contracting system"],
    max_lineas=2
)

procedimiento_contratacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Procedimiento de contratación", "Contracting procedure"],
    max_lineas=2
)

tipo_tramitacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Tipo de tramitación", "Processing type"],
    max_lineas=2
)

metodo_presentacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Método de presentación de la oferta", "Method of presentation of the offer"],
    max_lineas=2
)

fecha_fin_presentacion = extraer_bloque_despues_etiqueta(
    lineas_html,
    ["Fecha fin de presentación de oferta", "Deadline for submission of tenders"],
    max_lineas=2
)

In [37]:
# ============================================================
# 5. Buscar etiquetas reales que faltan
# ============================================================

for busqueda in [
    "Evaluación",
    "Evaluation",
    "80.366",
    "Euros",
    "Estimated",
    "Valor",
    "Estado",
    "Tender",
]:
    print("\n" + "=" * 80)
    print("Búsqueda:", busqueda)

    encontrados = [
        (i, linea)
        for i, linea in enumerate(lineas_html)
        if busqueda.lower() in linea.lower()
    ]

    for i, linea in encontrados[:20]:
        print(f"{i}: {linea}")


Búsqueda: Evaluación
51: Evaluación

Búsqueda: Evaluation

Búsqueda: 80.366
58: 80.366,00

Búsqueda: Euros
56: Euros
59: Euros

Búsqueda: Estimated
57: Estimated value of the contract

Búsqueda: Valor
102: Informe de valoración de los criterios de adjudicación cuantificables mediante juicio de valor

Búsqueda: Estado
49: https://contrataciondelestado.es/wps/poc?uri=deeplink:detalle_licitacion&idEvl=rsl%2BfImh9a6LAncw3qdZkA%3D%3D
109: https://contrataciondelestado.es/wps/poc?uri=deeplink:detalle_licitacion&idEvl=rsl%2BfImh9a6LAncw3qdZkA%3D%3D

Búsqueda: Tender
38: Details of tender
50: State of the Tender
